# Cancer Risk Assessment — Exploratory Data Analysis

**Dataset**: Cancer Patient Data (Kaggle)  
**Target**: `Level` — Low / Medium / High cancer risk  
**Goal**: Understand distributions, correlations, class imbalance, and feature importance before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

DATA_PATH = '../data/raw/cancer_patient_data.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Info & Missing Values

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Duplicates: {df.duplicated().sum()} ===')

In [ ]:
# Drop ID columns before analysis
drop_cols = [c for c in df.columns if c.lower() in ['index', 'patient id']]
df = df.drop(columns=drop_cols)
print(f'Dropped: {drop_cols}')
print(f'Working shape: {df.shape}')
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Target Distribution — Class Imbalance Check

In [ ]:
TARGET = 'Level'
label_order = ['Low', 'Medium', 'High']
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

counts = df[TARGET].value_counts().reindex(label_order)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors[l] for l in counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            colors=[colors[l] for l in counts.index],
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Cancer Risk Level', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../artifacts/plots/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f'Imbalance ratio (max/min): {imbalance_ratio:.2f}x')
print(counts)

## 3. Feature Distributions by Risk Level

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET]
print(f'{len(feature_cols)} features: {feature_cols}')

In [ ]:
# KDE plots for top features
top_features = ['Age', 'Smoking', 'Genetic Risk', 'Air Pollution',
                'Alcohol use', 'Obesity', 'Chest Pain', 'Fatigue']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top_features):
    for label in label_order:
        subset = df[df[TARGET] == label][feat]
        subset.plot.kde(ax=ax, label=label, color=colors[label], linewidth=2)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Risk Level', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../artifacts/plots/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_enc = df.copy()
le = LabelEncoder()
df_enc[TARGET] = le.fit_transform(df_enc[TARGET])

corr = df_enc.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../artifacts/plots/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Correlation with Target

In [ ]:
target_corr = df_enc.corr()[TARGET].drop(TARGET).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 9))
bar_colors = ['#e74c3c' if v > 0 else '#3498db' for v in target_corr.values]
target_corr.plot.barh(ax=ax, color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Risk Level')
ax.set_title('Feature Importance (Correlation with Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../artifacts/plots/eda_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 positive correlators:')
print(target_corr.tail(5))
print('\nTop 5 negative correlators:')
print(target_corr.head(5))

## 6. SMOTEENN Effect — Before vs After

In [ ]:
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

X = df_enc.drop(columns=[TARGET]).values.astype(float)
y = df_enc[TARGET].values

X_sc = StandardScaler().fit_transform(X)

smoteenn = SMOTEENN(smote=SMOTE(k_neighbors=5, random_state=42), random_state=42)
X_res, y_res = smoteenn.fit_resample(X_sc, y)

before = pd.Series(y).value_counts().sort_index()
after  = pd.Series(y_res).value_counts().sort_index()

label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
before.index = [label_map[i] for i in before.index]
after.index  = [label_map[i] for i in after.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, data, title in [(axes[0], before, 'Before SMOTEENN'),
                        (axes[1], after,  'After SMOTEENN')]:
    bars = ax.bar(data.index, data.values,
                  color=[colors[l] for l in data.index], edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Sample Count')

plt.suptitle('Class Imbalance: Before vs After SMOTEENN', fontsize=14)
plt.tight_layout()
plt.savefig('../artifacts/plots/eda_smoteenn_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Before — Total: {len(y)}')
print(before)
print(f'\nAfter  — Total: {len(y_res)}')
print(after)

## 7. EDA Summary

| Insight | Finding |
|---|---|
| Class imbalance | Moderate — SMOTEENN recommended |
| Top risk drivers | Smoking, Genetic Risk, Air Pollution, Obesity |
| Protective factors | Balanced Diet (negative correlation) |
| Missing values | None |
| Scaling needed | Yes — features on different numeric scales |
| Ready for modelling | ✅ |